# Lab 07 · Observability, tracing, and swappable agents

**Prepared by Mustafa Assaf** · Lecture 3: Debate, trace, decide · ~15–20 min core work (estimate; setup and extras take longer)

## Start with a small example

First learn the mechanism with an everyday example. Then apply it to the software architecture exercise below.

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Choose `PROVIDER` (`"freeinference"`, `"gemini"`, or `"deepseek"`), and save its key in Colab Secrets (`FREEINFERENCE_API_KEY`, `GEMINI_API_KEY`, or `DEEPSEEK_API_KEY`). `ask` stops after 100 calls; rerun setup to reset the count.

In [ ]:
from importlib.util import find_spec
from importlib.metadata import PackageNotFoundError, version
try:
    coursekit_version = version("agentic-ai-coursekit")
except PackageNotFoundError:
    coursekit_version = None
if coursekit_version != "0.5.1" or not all(find_spec(name) for name in ['coursekit', 'langsmith']):
    %pip install -q --upgrade "agentic-ai-coursekit[framework] @ https://github.com/mkassaf/agentic-ai-course/archive/a0b8f10a4139527a62c3a566286b9086f4aea7fe.zip"
    import importlib, sys
    for name in list(sys.modules):
        if name == "coursekit" or name.startswith("coursekit."):
            del sys.modules[name]
    importlib.invalidate_caches()
from coursekit import make_ask
PROVIDER = "deepseek"  # or "gemini" or "freeinference"
ask = make_ask(provider=PROVIDER)

## Warm-up · Follow a decision back to its source

The travel planner cites message `m1` when it chooses the museum. Run the cell and follow that ID back to the note about the closed park. The council exercise uses the same idea to connect a decision to its proposals and source notes.

In [ ]:
source = {"id": "m1", "sender": "guide", "text": "The park is closed today."}
decision = {"id": "m2", "sender": "planner", "text": "Visit the museum.", "cites": "m1"}
messages = {source["id"]: source, decision["id"]: decision}
print("Decision:", decision["text"])
print("Based on:", messages[decision["cites"]]["text"])

**Next:** Next, follow an architecture council's messages and evidence through structured logs.

## Follow a council's decisions
This lab uses the same kind of ShopEasy support search as Lab 06. Two model reviewers recommend a design. You will save what they said and follow the final choice back to its supporting notes.

**Observability** means being able to tell what a program did from the records it produced. The notebook shows each record on screen and saves a copy to `council.log`. A **message ID** identifies one record. A **trace ID** connects the records from one run, so you can tell them apart from a later run.

The **registry** is the dictionary listing the reviewers. Add a Python-rule reviewer to that dictionary, then rerun the same collection and voting steps. The main exercise uses your regular model key. The optional LangSmith cell uses a separate key.

**At a glance**

![Lab 07 at a glance: Minimalist (model agent) or Modernizer (model agent) or Rulebook (added in one line), then Schema check (structured message), then Write the log (JSON + trace_id), then Live log (same JSON lines) or council.log (one JSON per line) or LangSmith (trace tree (optional)), then Chair (majority or escalate), then Evidence trail (ids from council.log).](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/07.svg)

## Think about the example

An agent review contains several messages about one decision. A trace ID groups the run; a message ID identifies one message. A source link records what a claim is based on.

**Concepts:** Correlation IDs across service boundaries; Structured logs and immutable event facts.

Use the scenario above to consider these questions before running the code:

1. What is the difference between a trace ID and a message ID?
2. Why does an event or agent message need both identity and provenance?

## The idea in context

Observability makes a workflow inspectable during and after a run. Each proposal is logged, and a registry lists reviewer settings.

Agent-run traces and microservice request traces share correlation principles but are different systems.

| Key term | Meaning |
| --- | --- |
| **Observability** | Seeing what a running system does from its outputs: logs, metrics, and traces. |
| **Trace** | A record of one run as a tree of steps, with each step's input, output, and timing; LangSmith shows traces of LLM applications. |
| **Log** | A time-stamped record of what happened; structured logs, such as JSON lines, can be searched and counted. |
| **Trace ID** | A label shared by one run's messages so they can be found and compared together; a message ID identifies a single message. |
| **Registry** | A lookup table of agents by name, so the system never hard-codes who takes part. |

## Pattern · Shared Memory

*Memory pattern from the [Architectural Patterns for Agentic AI](https://karthikv1392.github.io/agentic-patterns/) field guide. Related: Registry & Adapter.*

| | |
| --- | --- |
| **Context** | Agents may work in separate contexts but still need a common source of truth for goals, progress, and handoffs. |
| **Problem** | Copying messages between agents duplicates context and creates inconsistent versions of the shared state. |
| **Solution** | Give the agents one shared memory, so that what one agent writes is immediately available to the others. |
| **Use when** | Agents share a project state; their own contexts should stay separate; a common task board is useful. |
| **Trade-off** | Reduces duplication, but concurrent updates need clear ownership to avoid overwrites and races. |
| **Flow** | Agent A → Shared memory → Agent B → Coordinated state |
| **Quality attributes** | Benefits: Interoperability, Reliability, Cost. Liabilities: Reliability, Security, Maintainability. |
| **In this lab** | The local log is a shared record for the chair and students. The reviewers do not automatically receive previous messages as context. This single-process example does not implement a concurrent or tamper-proof shared-memory service. |

## Main exercise at a glance

1. Prepare the case and reviewer settings
2. Set up a live log and a file
3. Collect and log proposals
4. Count valid proposals and log a decision
5. Follow the log and add a reviewer

## Step 1 · Prepare the case and reviewer settings
`CASE` describes what ShopEasy needs. `SOURCES` gives short notes to cite, and `OPTIONS` lists the choices the chair can accept.

Each entry in the `council` registry has a name, a `kind` that says how to get an answer, and a `stance` that guides the model's recommendation. Look for `Minimalist` and `Modernizer` in the output. Both start as model reviewers.

In [ ]:
import itertools, json, logging, sys
from collections import Counter
from datetime import datetime, timezone
from langsmith import trace

SOURCES = {"R2": "API composition fits small lookups by id; broad search can require large joins.",
           "R3": "CQRS supports rich queries but adds a stale read model.",
           "C1": "ShopEasy support searches orders and payments by name."}
CASE = {"name": "ShopEasy support", "cross_service_search": True}
OPTIONS = ["API composition", "CQRS"]
council = {
    "Minimalist": {"kind": "model", "stance": "Prefer the simplest design that works."},
    "Modernizer": {"kind": "model", "stance": "Prefer a read model for growing search traffic."},
}
ids = itertools.count()
run_ids = itertools.count(1)
print("Reviewers:", list(council))

def parse_proposal(text):
    try:
        body = json.loads(text)
    except json.JSONDecodeError:
        return {}, "Invalid JSON"
    if not isinstance(body, dict):
        return {}, "Expected a JSON object"
    return body, None

## Step 2 · Set up a live log and a file
A **structured log** gives each record the same named fields, such as sender, time, and trace ID. Here each line is one JSON object. The logger displays a copy in the notebook and saves the same line in `council.log`.

Look for `Log ready: council.log`. In Colab, open the Files sidebar to find the file. Running Step 2 clears the previous file, so skip this step when you want to compare runs in the same log.

In [ ]:
logger = logging.getLogger("council")
logger.setLevel(logging.INFO)
live = logging.StreamHandler(sys.stdout)
to_file = logging.FileHandler("council.log", mode="w")
formatter = logging.Formatter("%(message)s")
live.setFormatter(formatter)
to_file.setFormatter(formatter)
logger.handlers = [live, to_file]
logger.propagate = False
print("Log ready: council.log")

## Optional · Turn on LangSmith tracing
You can skip this cell and go straight to Step 3. The local log already records proposals and decisions. LangSmith is a separate service that can display the council run as a group of reviewer calls, along with inputs and outputs.

To try it, save `LANGSMITH_API_KEY` in Colab Secrets, grant this notebook access, and run the optional cell. It sends the classroom case and model replies to LangSmith when you run Step 3. Keep personal and confidential data out of the example.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
    os.environ["LANGSMITH_TRACING"] = "true"
    print("LangSmith tracing is on for Steps 3–5.")
except Exception:
    print("No LANGSMITH_API_KEY found; local logging only.")

## Step 3 · Collect and log proposals
Python reads the registry and asks each reviewer for a recommendation. A model reviewer gets a prompt with the case and sources. A rule reviewer answers from an `if` condition once you add it in Step 5.

Look at each printed log record. It shows the reviewer name, the option they chose, and the source IDs they cited. They count as a `proposal` if the option is allowed, there is a claim, and every citation is a known source. Empty citation lists, unknown choices, and malformed replies are logged as `rejected` with a reason. A real citation gives you a source to inspect.

The records from this run all have the same `trace_id`. The loop moves on to the next reviewer after a rejection. Step 4 can use the records from reviewers whose replies passed.

In [ ]:
accepted = []
trace_id = f"support-review-{next(run_ids)}"
with trace("council", inputs={"case": CASE}) as council_trace:
    for name, reviewer in council.items():
        with trace(name, inputs={"case": CASE}) as reviewer_trace:
            parse_error = None
            if reviewer["kind"] == "rule":
                option = "CQRS" if CASE["cross_service_search"] else "API composition"
                source = "R3" if CASE["cross_service_search"] else "R2"
                reply = {"option": option, "claim": "Review this read pattern and its cost.", "cites": [source, "C1"]}
            else:
                prompt = (f"You are {name}. {reviewer['stance']}\nSources: {json.dumps(SOURCES)}\n"
                          f"Case: {json.dumps(CASE)}\nReply with JSON only, no code fences: "
                          '{"option":"API composition or CQRS","claim":"one sentence","cites":["source ids"]}')
                reply, parse_error = parse_proposal(ask(prompt))
            claim, cites = reply.get("claim"), reply.get("cites", [])
            valid = (reply.get("option") in OPTIONS and isinstance(claim, str) and bool(claim.strip())
                     and isinstance(cites, list) and bool(cites)
                     and all(isinstance(source_id, str) and source_id in SOURCES for source_id in cites))
            message = {"id": f"m{next(ids)}", "sender": name, "kind": "proposal" if valid else "rejected",
                       "option": str(reply.get("option")), "claim": str(claim),
                       "cites": cites, "trace_id": trace_id, "time": datetime.now(timezone.utc).isoformat()}
            if not valid:
                message["reason"] = parse_error or "Unknown option, missing claim, or invalid citations"
            logger.info(json.dumps(message))
            reviewer_trace.end(outputs={"proposal": message})
            if valid:
                accepted.append(message)
    council_trace.end(outputs={"accepted": accepted})

## Step 4 · Count valid proposals and log a decision
The chair is Python code. It declares a decision when more than half of the registered reviewers choose the same option. With two reviewers, both must agree. When the threshold is missed, the chair records `escalation` and prints `no majority` as the option.

The chair records the same trace ID as the proposals in Step 3. Its `cites` field points to the proposal message IDs used in the outcome. The proposals' own `cites` fields refer to source IDs such as `R3`.

In [ ]:
votes = Counter()
for message in accepted:
    votes[message["option"]] += 1
option, count = votes.most_common(1)[0] if votes else (None, 0)
decided = 2 * count > len(council)
citations = []
for message in accepted:
    if not decided or message["option"] == option:
        citations.append(message["id"])
decision = {"id": f"m{next(ids)}", "sender": "Chair", "kind": "decision" if decided else "escalation",
            "option": option if decided else "no majority", "claim": f"{count} of {len(council)} reviewers",
            "cites": citations, "trace_id": trace_id, "time": datetime.now(timezone.utc).isoformat()}
logger.info(json.dumps(decision))
print("Decision:", decision)

## Step 5 · Follow the log and add a reviewer
Start with the decision ID and trace ID. The code reads `council.log`, finds the cited proposal records from that run, and prints the source text each proposal cited. Read the output as decision → proposal → sources.

This cell also adds `Rulebook` to the registry. Rerun Steps 3–5 to include it in the vote. Compare the two trace IDs in the log and check for a line from Rulebook in the second run. Running Steps 3–5 appends records to the existing log. To start from the original two reviewers with an empty log, rerun Steps 1 and 2.

In [ ]:
saved_messages = []
with open("council.log") as log_file:
    for line in log_file:
        saved_messages.append(json.loads(line))
print("Run:", decision["trace_id"], "| Decision:", decision["id"], decision["option"])
for proposal_id in decision["cites"]:
    for message in saved_messages:
        if message["trace_id"] == decision["trace_id"] and message["id"] == proposal_id:
            print("Proposal:", message["sender"], message["claim"])
            for source_id in message["cites"]:
                print("  Source:", source_id, SOURCES[source_id])
council["Rulebook"] = {"kind": "rule", "stance": "Use a simple search rule."}
print("Added Rulebook. Rerun Steps 3–5 to include it.")

## What you should see
The first run has records for two reviewers and one chair outcome. Steps 3–5 repeated after adding Rulebook give you three reviewer records and a new outcome with a new trace ID. Results can be decisions or escalations depending on the model replies.

## Takeaway
Structured logs let you look up what happened in one run and see the evidence behind the result. A registry lets the same collection loop work with both model and rule reviewers.

## Exercise · Trace a change in the council's input

1. After Step 5 has added Rulebook, add a code cell and set `CASE["cross_service_search"] = False` and `SOURCES["C1"] = "ShopEasy support looks up one order by ID."` Predict which option Rulebook will choose, then run your new cell.
2. Rerun Steps 3–5. The log keeps both runs when you leave Step 2 alone. Compare Rulebook's logged `option` and `cites` with your prediction.
3. In `council.log`, use the new `trace_id` to find the chair's outcome. Pick one ID in its `cites` list and follow it to the proposal and source text.

**More to try**

- Compare `trace_id` and `id` across two records. Which field identifies a whole run and which identifies a single message?
- Change one model reviewer's `stance` in the registry and rerun Steps 3–5. Use that run's trace ID to locate the new outcome.